In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from strategies import VAR


notebook_file = globals().get("__vsc_ipynb_file__")
NOTEBOOK_DIR = Path(notebook_file).resolve().parent if notebook_file else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))



DATE = "AsOfDate"



prices = pl.read_csv(NOTEBOOK_DIR / "../prices/example_prices.csv", try_parse_dates=True).rename({"dates": DATE})
instruments = [c for c in prices.columns if c != DATE]
ret = prices.select(DATE, pl.col(instruments).diff())

In [17]:
model = VAR.VARmodel(num_vars=5, max_lag=10)
model.load_training_data(ret[instruments].to_numpy()[1:])  # Exclude the first row since it contains NaN values due to differencing



In [6]:

print(model.training_data.data.shape)
print(model.training_data.validation.shape)

(5, 5944, 10)
(5944, 5)


In [18]:
model.fit()

In [20]:
print(model.predictors)

[array([[-1.75532558e+00, -1.17405430e+00, -3.34432923e-01,
        -2.45169839e+00,  5.71342823e+00],
       [-2.64135937e-02,  1.68214996e-02,  1.97137698e-02,
         2.91215036e-03,  3.02875366e-02]]), array([[-5.56970618e+00, -2.56704537e+00, -3.17210572e+00,
         7.24117266e+00,  4.07163161e+00],
       [ 5.92122749e-01,  2.65477975e-01,  1.18565111e+01,
        -8.53735145e+00, -4.17883407e+00],
       [-2.63130194e-02,  1.67528992e-02,  1.96382562e-02,
         2.89913119e-03,  3.01615476e-02]]), array([[ 1.23376320e+00, -2.44834444e+00,  4.59609248e+00,
        -2.64931859e+00, -7.31075930e-01],
       [ 2.50541057e+00, -2.03529904e+00, -9.02561977e-01,
         2.87416558e+00, -2.43776550e+00],
       [-2.07583783e+00, -2.80422731e+00,  4.55228569e+00,
        -4.12500221e+00,  4.45070813e+00],
       [-2.62856015e-02,  1.67337604e-02,  1.96161077e-02,
         2.89574529e-03,  3.01258796e-02]]), array([[ 4.66803992e-01,  7.60372536e-01, -4.70906796e-01,
        -1.31097